In [ ]:
import os
import glob
import sys
import subprocess
from pathlib import Path
script_path = Path("/workspace/script")
if str(script_path) not in sys.path:
    sys.path.append(str(script_path))
import rois2mask

In [ ]:
# retrieve the paths to the crop stack images for running crop2bitwise.py
# recursively search the directories to find Stack.tif and return its absolute path
path_dir = "../demo_image"
sub_dir = "extract_image"
stack_file = "Stack.tif"
list_path = rois2mask.find_all_file(path_dir, sub_dir, stack_file)
list_path


Stack.tif not found in demo_image/Project000_YSIY884/CPsegm_PSM/crop_Series015_480min_Lng_SVCC_Processed001/extract_image
Stack.tif not found in demo_image/Project000_YSIY884/CPsegm_PSM/crop_Series015_480min_Lng_SVCC_Processed001_ch00/extract_image
Stack.tif not found in demo_image/Project001_YSIY1448/CPsegm_PSM/crop_Series010_450min_Lng_SVCC_Processed001/extract_image
Stack.tif not found in demo_image/Project001_YSIY1448/CPsegm_PSM/crop_Series010_450min_Lng_SVCC_Processed001_ch00/extract_image
Stack.tif not found in demo_image/Project001_YSIY1448/CPsegm_PSM/crop_Series010_450min_Lng_SVCC_Processed001_ch01/extract_image


['demo_image/Project000_YSIY884/CPsegm_PSM/crop_Series015_480min_Lng_SVCC_Processed001_ch01/extract_image/Stack.tif']

In [3]:
# create mask images and copy images with matching filenames into train_image
roi_file = "RoiSet.zip"
list_path_roizip = rois2mask.find_all_file(path_dir, sub_dir, roi_file)

for path in list_path_roizip:
    rois2mask.rois2mask(path, sub_dir, stack_file)

RoiSet.zip not found in demo_image/Project000_YSIY884/CPsegm_PSM/crop_Series015_480min_Lng_SVCC_Processed001/extract_image
RoiSet.zip not found in demo_image/Project000_YSIY884/CPsegm_PSM/crop_Series015_480min_Lng_SVCC_Processed001_ch00/extract_image
RoiSet.zip not found in demo_image/Project001_YSIY1448/CPsegm_PSM/crop_Series010_450min_Lng_SVCC_Processed001/extract_image
RoiSet.zip not found in demo_image/Project001_YSIY1448/CPsegm_PSM/crop_Series010_450min_Lng_SVCC_Processed001_ch00/extract_image
RoiSet.zip not found in demo_image/Project001_YSIY1448/CPsegm_PSM/crop_Series010_450min_Lng_SVCC_Processed001_ch01/extract_image


### Copy all `mask_image` and `extract_image` to a single dir
### Devide into 80% train, 10% val, 10% test

In [ ]:
# configure the target directories
target_dir = "../demo_image/DataSet"
target_images = os.path.join(target_dir, "images")
target_masks = os.path.join(target_dir, "masks")

os.makedirs(target_images, exist_ok=True)
os.makedirs(target_masks, exist_ok=True)

images_train = os.path.join(target_images, "train")
os.makedirs(images_train, exist_ok=True)

masks_train = os.path.join(target_masks, "train")
os.makedirs(masks_train, exist_ok=True)

images_val = os.path.join(target_images, "val")
os.makedirs(images_val, exist_ok=True)

masks_val = os.path.join(target_masks, "val")
os.makedirs(masks_val, exist_ok=True)

images_test = os.path.join(target_images, "test")
os.makedirs(images_test, exist_ok=True)

masks_test = os.path.join(target_masks, "test")
os.makedirs(masks_test, exist_ok=True)

In [16]:
strain_list = [d for d in os.listdir(path_dir) 
               if d.startswith("Project") and os.path.isdir(os.path.join(path_dir, d))]

for strain in strain_list:
    path_with_strain = os.path.join(path_dir, strain, "CPsegm_PSM")
    img_list = [d_img for d_img in os.listdir(path_with_strain) if os.path.isdir(os.path.join(path_with_strain, d_img)) and "Series" in d_img]
    for img in img_list:
        path_with_image = os.path.join(path_with_strain, img)
        path_mask = os.path.join(path_with_image, "mask_image")
        # copy every .png inside mask_image into target_mask
        png_mask = glob.glob(os.path.join(path_mask, "*.png"))
        for png in png_mask:
            subprocess.run(["cp", png, target_masks])
        
        path_train = os.path.join(path_with_image, "train_image")
        png_train = glob.glob(os.path.join(path_train, "*.png"))
        for png in png_train:
            subprocess.run(["cp", png, target_images])


In [ ]:
import random
import shutil
from pathlib import Path
from collections import defaultdict

# --- Configuration ---
SOURCE_DATA_DIR = Path("../demo_image/DataSet")
DESTINATION_DIR = Path("../demo_image/DataSet")
SPLIT_RATIOS = {"train": 0.8, "val": 0.1, "test": 0.1}
RANDOM_SEED = 42

# --- Logic based on your final instruction ---

def create_file_map(source_images_dir: Path, source_masks_dir: Path) -> dict:
    """
    Creates a map by using the image's own basename as the key,
    and finding all mask files that start with that key.
    This does NOT group by '_chXX'.
    """
    image_paths = list(source_images_dir.glob("*.png"))
    image_basenames = {p.stem for p in image_paths}
    
    # The map structure will be:
    # { 'image_basename': {'image': Path, 'masks': [Path, ...]} }
    file_map = {basename: {"image": None, "masks": []} for basename in image_basenames}
    
    for img_path in image_paths:
        file_map[img_path.stem]["image"] = img_path

    mask_paths = list(source_masks_dir.glob("*.png"))
    for mask_path in mask_paths:
        for basename in image_basenames:
            if mask_path.name.startswith(basename):
                file_map[basename]["masks"].append(mask_path)
                break
    
    valid_map = {}
    for basename, paths in file_map.items():
        if paths["image"] and paths["masks"]:
            valid_map[basename] = paths
            
    return valid_map

def split_and_copy_files(file_map: dict, destination_base_dir: Path):
    """Splits and copies files into train/val/test sets."""
    # The unit to be shuffled is the image basename.
    group_keys = list(file_map.keys())
    random.seed(RANDOM_SEED)
    random.shuffle(group_keys)

    total_keys = len(group_keys)
    train_end = int(total_keys * SPLIT_RATIOS["train"])
    val_end = train_end + int(total_keys * SPLIT_RATIOS["val"])

    splits = {
        "train": group_keys[:train_end],
        "val": group_keys[train_end:val_end],
        "test": group_keys[val_end:],
    }

    for split_name, keys in splits.items():
        output_images_dir = destination_base_dir / "images" / split_name
        output_masks_dir = destination_base_dir / "masks" / split_name
        output_images_dir.mkdir(parents=True, exist_ok=True)
        output_masks_dir.mkdir(parents=True, exist_ok=True)

        for key in keys:
            # key is an image basename, e.g., 'Project...2_0_crop_region_extract'
            
            # Copy the single image file. Access with 'image' (singular).
            image_to_copy = file_map[key]['image']
            shutil.copy(image_to_copy, output_images_dir)

            # Copy all corresponding mask files. Access with 'masks' (plural).
            for mask_to_copy in file_map[key]['masks']:
                shutil.copy(mask_to_copy, output_masks_dir)

def main():
    """Main execution function."""
    source_images_dir = SOURCE_DATA_DIR / "images"
    source_masks_dir = SOURCE_DATA_DIR / "masks"

    if not (source_images_dir.is_dir() and source_masks_dir.is_dir()):
        return

    file_map = create_file_map(source_images_dir, source_masks_dir)
    
    if file_map:
        split_and_copy_files(file_map, DESTINATION_DIR)

if __name__ == "__main__":
    main()

## COCO annnotation


In [ ]:
!python ./create_coco_annotations.py ./demo_image/DataSet ./demo_image/DataSet


## Config writer

In [ ]:
from mmengine import Config
from mmengine.runner import set_random_seed

config_from = "../mmdetection/configs/mask_rcnn/mask-rcnn_r50_fpn_2x_coco.py"


def config_writer(data_root, config_from, train_ann, val_ann, test_ann, work_dir, config_path):
    cfg = Config.fromfile(config_from)
    # Modify dataset classes and color
    cfg.metainfo = {
        'classes': ('PSM', )
        # 'palette': [
        #     (220, 20, 60),
        # ]
    }

    # Modify dataset type and path
    cfg.data_root = data_root
    
    cfg.train_dataloader.dataset.ann_file = train_ann
    cfg.train_dataloader.dataset.data_root = cfg.data_root
    cfg.train_dataloader.dataset.data_prefix.img = 'images/train/'
    cfg.train_dataloader.dataset.metainfo = cfg.metainfo

    cfg.train_dataloader.batch_size = 4

    cfg.val_dataloader.dataset.ann_file = val_ann
    cfg.val_dataloader.dataset.data_root = cfg.data_root
    cfg.val_dataloader.dataset.data_prefix.img = 'images/val/'
    cfg.val_dataloader.dataset.metainfo = cfg.metainfo

    cfg.test_dataloader.dataset.ann_file = test_ann
    cfg.test_dataloader.dataset.data_root = cfg.data_root
    cfg.test_dataloader.dataset.data_prefix.img = 'images/test/'
    cfg.test_dataloader.dataset.metainfo = cfg.metainfo
    
    cfg.work_dir = './' + work_dir

    # Modify metric config
    cfg.val_evaluator.ann_file = cfg.data_root+'/'+ val_ann
    cfg.val_evaluator.outfile_prefix = cfg.work_dir + '/val_results'
    cfg.test_evaluator.ann_file = cfg.data_root+'/'+ test_ann
    cfg.test_evaluator.outfile_prefix = cfg.work_dir + '/test_results'

    # Modify num classes of the model in box head and mask head
    cfg.model.roi_head.bbox_head.num_classes = 1
    cfg.model.roi_head.mask_head.num_classes = 1


    # We can set the evaluation interval to reduce the evaluation times
    cfg.train_cfg.val_interval = 3
    # We can set the checkpoint saving interval to reduce the storage cost
    cfg.default_hooks.checkpoint.interval = 3

    # The original learning rate (LR) is set for 8-GPU training.
    # We divide it by 8 since we only use one GPU.
    cfg.optim_wrapper.optimizer.lr = 0.02 / 8
    cfg.default_hooks.logger.interval = 10


    # Set seed thus the results are more reproducible
    cfg.seed = 0
    set_random_seed(0, deterministic=False)

    # We can also use tensorboard to log the training process
    cfg.vis_backends[0] = dict(type='TensorboardVisBackend')
    
    cfg.visualizer.vis_backends[0] = dict(type='TensorboardVisBackend')

    #------------------------------------------------------
    config=config_path
    with open(config, 'w') as f:
        f.write(cfg.pretty_text)

In [ ]:
train_ann = 'train_annotations.json'
val_ann = 'val_annotations.json'
test_ann = 'test_annotations.json'


work_dir = "../demo_image/Results_Train"
dataset_root = "../demo_image/DataSet"
config_path = "../mmdetection/configs/custom_PSM/mask-rcnn_r50_fpn_2x_coco_PSM.py"

config_dir = os.path.dirname(config_path)
os.makedirs(config_dir, exist_ok=True)

config_writer(dataset_root, config_from, train_ann, val_ann, test_ann, work_dir, config_path)

## Train
execute the following command after activating the venv by `source .venv_DeMemSeg/bin/activate`:

```bash
python mmdetection/tools/train.py mmdetection/configs/custom_PSM/mask-rcnn_r50_fpn_2x_coco_PSM.py
```

In [ ]:
# load tensorboard in jupyter notebook
%load_ext tensorboard

In [ ]:
# see curves in tensorboard
# if you see <IPython.core.display.HTML object> please run it again
%tensorboard --logdir ./PSM_exps/20240704_234417